# Sparse Tensors and CNN-based Analysis

In this notebook, we'll walk through the process of converting simulated output to a [spconv](https://github.com/traveller59/spconv) SparseTensor object, and show a simple sparse CNN-based regression model

In [1]:
from gampixpy import config, detector, generator, analysis, output

In [2]:
# set up matplotlib for this notebook
%matplotlib widget

# import torch and set the default device (needed for GPU)
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    # Set the default device to CUDA
    torch.set_default_device(device)
    print(f"Default device set to: {torch.cuda.get_device_name(device)}")
else:
    device = torch.device('cpu')
    print("CUDA is not available, using CPU")

import spconv.pytorch as spconv

CUDA is not available, using CPU


In [3]:
# the `demo_large_pixels` readout config is nice for visualizing
# but it will often not produce enough pixel hits.
# use whichever suits your purpose!
conf = config.ConfigManager(detector_config = 'far_detector_vd',
                            #readout_config = 'demo_large_pixels',
                            readout_config = 'GAMPixD',
                            )

## We can also build one of these from a yaml file
## The configs which come along with the library are
## accessible from the `preset_[...]_configs`, but can
## also be found alongside the library code
# readout_config = config.ReadoutConfig('../../gampixpy/readout_config/GAMPixD_notruth.yaml')

detector_model = detector.DetectorModel(config_manager=conf)

ps_gen = generator.PointSource(x_range = [-10, 10],
                               y_range = [-10, 10],
                               z_range = [-10, 10],
                               t_range = [0, 0],
                               q_range = [1e6, 1e8],
                              )

# clean the old output
!rm test_output.h5
# re-initialize the output manager
om = output.OutputManager('test_output.h5', config_manager=conf)

# for now, we'll do 10 events
n_events = 10

# let's add a nice progress bar
import tqdm
for i in tqdm.tqdm(range(n_events)):
    # generate a new point source object
    event_data = ps_gen.get_sample()
    event_meta = ps_gen.get_meta()

    # use `verbose = False` to suppress the info messages
    detector_model.simulate(event_data, verbose = False)

    # write this to the output file
    om.add_entry(event_data, event_meta)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.63it/s]


## `SparseTensorConverter`
This is just an `OutputParser` subclass which provides a method for generating sparse tensors from the `spconv` package.
It also allows for generating a random sampling order and an iterable interface so that it can be used as a dataloader object.

In [4]:
stc = analysis.SparseTensorConverter('test_output.h5', batch_size = 3)

event_id = 1
depth = stc.get_vertex_depth(event_id=event_id)
st = stc.get_sparsetensor(event_id=event_id, label=0)

print (depth)
print (st)
print (st.indices)
print (st.features)

tensor(320.7437)
SparseConvTensor[shape=torch.Size([400, 4])]
tensor([[ 0,  0,  1,  1],
        [ 0,  0,  1,  2],
        [ 0,  0,  1,  3],
        ...,
        [ 0,  4,  2, 17],
        [ 0,  4,  2, 18],
        [ 0,  4,  2, 19]], dtype=torch.int32)
tensor([[-1052.2500,   -10.2500,  1999.2144,   -33.9991],
        [-1052.2500,   -10.2500,  1999.7144,   -28.8572],
        [-1052.2500,   -10.2500,  2000.2144,   561.1018],
        ...,
        [-1050.2500,    -9.7500,  2007.2144,    21.2171],
        [-1050.2500,    -9.7500,  2007.7144,   -19.7442],
        [-1050.2500,    -9.7500,  2008.2144,    19.3860]])


/home/dan/.local/lib/python3.10/site-packages/gampixpy/analysis.py:474: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:230.)
  source_point_exp = torch.tensor([event_meta['vertex x'],


In [5]:
# An example network using spconv
from torch import nn
class ExampleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = spconv.SparseSequential(
            spconv.SparseConv3d(4, 64, 3), # just like nn.Conv3d but don't support group
            nn.BatchNorm1d(64), # non-spatial layers can be used directly in SparseSequential.
            nn.ReLU(),
            #spconv.SubMConv3d(64, 64, 3, indice_key="subm0"),
            #nn.BatchNorm3d(64),
            #nn.ReLU(),
            # when use submanifold convolutions, their indices can be shared to save indices generation time.
            #spconv.SubMConv3d(64, 64, 3, indice_key="subm0"),
            #nn.BatchNorm3d(64),
            #nn.ReLU(),
            #spconv.SparseConvTranspose3d(64, 64, 3, 2),
            #nn.BatchNorm3d(64),
            #nn.ReLU(),
            #spconv.ToDense(), # convert spconv tensor to dense and convert it to NCHW format.
            #nn.Conv3d(64, 64, 3),
            #nn.BatchNorm3d(64),
            #nn.ReLU(),
        )
        
    def forward(self, x):
        return self.net(x)

In [6]:
net = ExampleNet()
print (st.features)
print (st.indices)
net.net(st)

tensor([[-1052.2500,   -10.2500,  1999.2144,   -33.9991],
        [-1052.2500,   -10.2500,  1999.7144,   -28.8572],
        [-1052.2500,   -10.2500,  2000.2144,   561.1018],
        ...,
        [-1050.2500,    -9.7500,  2007.2144,    21.2171],
        [-1050.2500,    -9.7500,  2007.7144,   -19.7442],
        [-1050.2500,    -9.7500,  2008.2144,    19.3860]])
tensor([[ 0,  0,  1,  1],
        [ 0,  0,  1,  2],
        [ 0,  0,  1,  3],
        ...,
        [ 0,  4,  2, 17],
        [ 0,  4,  2, 18],
        [ 0,  4,  2, 19]], dtype=torch.int32)


SparseConvTensor[shape=torch.Size([19, 64])]

### To get batches of inputs, use the `SparseTensorConverter` as an iterable:

In [7]:
for sparsetensor_batch, depth in stc:
    print (depth)
    y = net(sparsetensor_batch)

tensor([327.3910, 320.3216, 320.7437])
tensor([332.9244, 318.9030, 317.7834])
tensor([321.7892, 324.6227, 320.1949])
